# 📓 Notebook 22 — Industry Frameworks Overview---**Phase 7 · Industry Frameworks**> The LLM ecosystem moves fast. Knowing the top frameworks — their strengths, trade-offs, and when to use each — is a key interview differentiator.### What You'll Learn| Framework | What It Does ||-----------|-------------|| **LlamaIndex** | Data framework for RAG — indexing, retrieval, query engines || **CrewAI** | Multi-agent orchestration with role-based agents || **AutoGen** | Microsoft's multi-agent conversation framework || **Haystack** | End-to-end NLP pipelines (by deepset) || **Semantic Kernel** | Microsoft's SDK for AI app development || **DSPy** | Programmatic prompt optimization (Stanford) |### 🏗️ Build: Compare Frameworks Side-by-Side

## 1. Framework Landscape### The 2024-2025 LLM Framework Ecosystem```┌─────────────────────────────────────────────────────────────┐│                    LLM APPLICATION LAYER                     │├──────────────┬────────────────┬──────────────────────────────┤│   RAG-First  │  Agent-First   │  Full-Stack                  ││              │                │                              ││  LlamaIndex  │  CrewAI        │  LangChain + LangGraph       ││  Haystack    │  AutoGen       │  Semantic Kernel             ││              │  DSPy          │                              │├──────────────┴────────────────┴──────────────────────────────┤│                    OBSERVABILITY LAYER                        ││         LangSmith  ·  Langfuse  ·  Phoenix  ·  W&B          │├──────────────────────────────────────────────────────────────┤│                    LLM PROVIDER LAYER                        ││     OpenAI  ·  Google Gemini  ·  Anthropic  ·  Local LLMs   │└──────────────────────────────────────────────────────────────┘```> **Interview Tip:** You should be able to name 3-4 frameworks, explain their sweet spots, and recommend one based on a given requirement.

---## 2. LlamaIndex — The Data Framework for RAG### When to Use LlamaIndex vs LangChain| Feature | LlamaIndex | LangChain ||---------|-----------|-----------|| Sweet spot | RAG & data indexing | General LLM apps & agents || Data connectors | 160+ (LlamaHub) | 100+ (Integrations) || Index types | Multiple (vector, keyword, tree, KG) | Mainly vector || Query engine | Sophisticated built-in | DIY with chains || Agent support | Good (recent) | Excellent (LangGraph) || Learning curve | Moderate | Higher |

In [ ]:
# LlamaIndex RAG — Minimal Example# pip install llama-indextry:    from llama_index.core import VectorStoreIndex, SimpleDirectoryReader, Settings, Document    from llama_index.llms.openai import OpenAI as LlamaOpenAI    import os        # Configure LlamaIndex    Settings.llm = LlamaOpenAI(model="gpt-4o-mini", temperature=0)        # Create documents    documents = [        Document(text="Python is a high-level programming language created by Guido van Rossum in 1991. It emphasizes code readability."),        Document(text="LlamaIndex is a data framework for LLM applications. It excels at indexing and retrieval."),        Document(text="RAG combines retrieval with generation to ground LLM responses in external data."),    ]        # Build index (embeddings + vector store in one line!)    index = VectorStoreIndex.from_documents(documents)        # Query    query_engine = index.as_query_engine()    response = query_engine.query("What is LlamaIndex?")        print(f"🦙 LlamaIndex Response: {response}")    print(f"   Sources: {[n.metadata for n in response.source_nodes]}")    except ImportError:    print("ℹ️ LlamaIndex not installed. Install with: pip install llama-index")    print("\n📝 Key LlamaIndex Concepts:")    print("   • SimpleDirectoryReader — Load docs from a folder")    print("   • VectorStoreIndex — Auto-embed and index documents")    print("   • QueryEngine — RAG in one line: index.as_query_engine()")    print("   • SubQuestionQueryEngine — Decomposes complex queries")    print("   • RouterQueryEngine — Routes to different indices")

---## 3. CrewAI — Multi-Agent Orchestration### Core Concept: Agents with RolesCrewAI gives agents **roles, goals, and backstories** — like hiring a team.```pythonresearcher = Agent(    role="Senior Research Analyst",    goal="Find comprehensive data on the topic",    backstory="You have 10 years of research experience...",    tools=[search_tool],)```

In [ ]:
# CrewAI Example# pip install crewaitry:    from crewai import Agent, Task, Crew, Process        # Define agents with roles    researcher = Agent(        role="Senior Research Analyst",        goal="Find comprehensive, accurate information on the given topic",        backstory="You are a thorough researcher with 10 years of experience in technology analysis.",        verbose=True,        allow_delegation=False,    )        writer = Agent(        role="Technical Writer",        goal="Write clear, engaging technical content",        backstory="You are a skilled technical writer who makes complex topics accessible.",        verbose=True,        allow_delegation=False,    )        # Define tasks    research_task = Task(        description="Research the current state of LLM agent frameworks in 2025. Focus on key players, trends, and adoption.",        expected_output="A detailed research summary with key findings and data points.",        agent=researcher,    )        writing_task = Task(        description="Write a concise article summarizing the research findings on LLM agent frameworks.",        expected_output="A well-structured 200-word article.",        agent=writer,    )        # Create crew    crew = Crew(        agents=[researcher, writer],        tasks=[research_task, writing_task],        process=Process.sequential,        verbose=True,    )        # Run    result = crew.kickoff()    print(f"\n📝 Crew Result: {result}")    except ImportError:    print("ℹ️ CrewAI not installed. Install with: pip install crewai")    print("\n📝 Key CrewAI Concepts:")    print("   • Agent(role, goal, backstory) — Character-driven agents")    print("   • Task(description, agent) — Assigned work items")    print("   • Crew(agents, tasks, process) — Orchestrated team")    print("   • Process.sequential — Agents work in order")    print("   • Process.hierarchical — Manager delegates to agents")    print("   • allow_delegation=True — Agents can delegate to each other")

---## 4. Microsoft AutoGen — Multi-Agent Conversations### Core Concept: Agents that Talk to Each OtherAutoGen models agents as **conversational participants** that chat to solve problems.```┌──────┐     ┌──────────┐     ┌──────────┐│ User │────→│ Assistant│────→│ Executor ││      │←────│  Agent   │←────│  Agent   │└──────┘     └──────────┘     └──────────┘```

In [ ]:
# AutoGen Example# pip install autogen-agentchattry:    from autogen import ConversableAgent    import os        llm_config = {"model": "gpt-4o-mini", "api_key": os.getenv("OPENAI_API_KEY")}        # Assistant agent    assistant = ConversableAgent(        name="assistant",        system_message="You are a helpful AI assistant. Solve tasks carefully.",        llm_config=llm_config,    )        # User proxy (simulates user interaction)    user_proxy = ConversableAgent(        name="user_proxy",        is_termination_msg=lambda msg: "TERMINATE" in msg.get("content", ""),        human_input_mode="NEVER",        max_consecutive_auto_reply=3,    )        # Start conversation    user_proxy.initiate_chat(        assistant,        message="What are the top 3 benefits of using LLM agents in software development? Be concise.",    )    except ImportError:    print("ℹ️ AutoGen not installed. Install with: pip install autogen-agentchat")    print("\n📝 Key AutoGen Concepts:")    print("   • ConversableAgent — Base agent that can chat")    print("   • AssistantAgent — LLM-powered agent")    print("   • UserProxyAgent — Simulates user or runs code")    print("   • GroupChat — Multiple agents discussing together")    print("   • GroupChatManager — Orchestrates group conversations")    print("   • Nested chats — Agents within agents")

---## 5. Framework Comparison Matrix### Side-by-Side Interview Reference| Feature | LangChain | LlamaIndex | CrewAI | AutoGen | LangGraph ||---------|-----------|-----------|--------|---------|-----------|| **Primary Use** | General LLM apps | RAG & data | Multi-agent teams | Agent conversations | Stateful agent graphs || **RAG Quality** | Good | Excellent | Basic | Basic | Good (via LangChain) || **Agent Support** | Very good | Good | Excellent | Excellent | Best || **Multi-Agent** | Manual | Limited | Built-in roles | Built-in chat | Built-in graph || **State Management** | Basic | Index-based | Task-based | Chat history | Full graph state || **Persistence** | Via LangGraph | Built-in | Basic | Basic | Checkpointing || **Human-in-loop** | Via LangGraph | No | Yes | Yes | Yes || **Learning Curve** | Medium-High | Medium | Low | Medium | Medium || **Production Ready** | Yes | Yes | Growing | Growing | Yes || **Community** | Largest | Large | Growing fast | Large | Growing || **Best For** | Full-stack LLM apps | Data-heavy RAG | Quick multi-agent | Research/prototyping | Production agents |

In [ ]:
# Summary comparisonframeworks = {    "LangChain + LangGraph": {        "best_for": "Full-stack LLM applications with complex agent workflows",        "key_feature": "LCEL composition + graph-based state management",        "when_to_use": "Production apps needing chains, RAG, agents, and persistence",        "companies_using": "Many startups and enterprises",    },    "LlamaIndex": {        "best_for": "Data-heavy RAG applications",        "key_feature": "160+ data connectors, multiple index types",        "when_to_use": "When data ingestion and retrieval quality is the priority",        "companies_using": "Companies with large document corpora",    },    "CrewAI": {        "best_for": "Role-based multi-agent teams",        "key_feature": "Agent roles, goals, backstories — character-driven",        "when_to_use": "When you want quick multi-agent setup with clear role separation",        "companies_using": "Rapid prototyping of agent teams",    },    "AutoGen": {        "best_for": "Multi-agent conversations and code generation",        "key_feature": "Agents as conversational participants",        "when_to_use": "Research, code generation, complex multi-turn agent collaboration",        "companies_using": "Microsoft ecosystem, research labs",    },    "DSPy": {        "best_for": "Systematic prompt optimization",        "key_feature": "Compile prompts from examples — no manual prompt engineering",        "when_to_use": "When you want to optimize prompts programmatically on a dataset",        "companies_using": "Research teams, Stanford NLP group",    },    "Haystack": {        "best_for": "Production NLP pipelines",        "key_feature": "Pipeline-based architecture, strong search integration",        "when_to_use": "Enterprise search, question answering over large document sets",        "companies_using": "deepset (creator), enterprise customers",    },}print("🏗️ Framework Decision Guide")print("=" * 60)for name, info in frameworks.items():    print(f"\n🔧 {name}")    print(f"   Best for: {info['best_for']}")    print(f"   Key feature: {info['key_feature']}")    print(f"   Use when: {info['when_to_use']}")

---## 6. Choosing the Right Framework — Decision Tree```Need to build an LLM app?    │    ├── Is it primarily a RAG/search app?    │   ├── YES → LlamaIndex (or LangChain for more flexibility)    │   └── NO ↓    │    ├── Does it need stateful agents with persistence?    │   ├── YES → LangGraph    │   └── NO ↓    │    ├── Do you need multiple specialized agents?    │   ├── YES, with clear roles → CrewAI    │   ├── YES, with conversation → AutoGen    │   └── NO ↓    │    ├── Do you need to optimize prompts programmatically?    │   ├── YES → DSPy    │   └── NO ↓    │    ├── General LLM application?    │   └── LangChain (most mature, largest ecosystem)    │    └── Enterprise NLP pipeline?        └── Haystack```> **Golden Rule:** Don't over-engineer. Start with the simplest framework that meets your needs. You can always migrate later.

---## 📝 Interview Quiz — Industry Frameworks### Q1: Compare LangChain and LlamaIndex. When would you choose each?<details><summary>💡 Answer</summary>**LangChain:** General-purpose LLM application framework.- Best for: Full-stack LLM apps with chains, agents, tools, memory- Strength: Broadest ecosystem, most integrations, LangGraph for production agents- Weakness: Can be complex for simple RAG**LlamaIndex:** Data-focused framework for RAG.- Best for: Apps where data ingestion and retrieval quality is critical- Strength: 160+ data connectors, multiple index types (vector, keyword, tree, knowledge graph)- Weakness: Agent support is secondary to data capabilities**Choose LangChain when:** You're building a general agent system or need the full toolkit.**Choose LlamaIndex when:** Your primary challenge is connecting to diverse data sources and building the best possible retrieval.**Both together:** Many production systems use LlamaIndex for the RAG pipeline and LangChain/LangGraph for the agent layer.</details>### Q2: What is CrewAI's key innovation? How is it different from building agents with LangGraph?<details><summary>💡 Answer</summary>**CrewAI's innovation:** Role-based agent modeling.- Each agent has a `role`, `goal`, `backstory` — like hiring for a job- Natural mental model: "I need a researcher, a writer, and a reviewer"- Built-in delegation: agents can ask other agents for help**vs LangGraph:**- LangGraph is lower-level: you define nodes, edges, state- LangGraph gives more control and flexibility- CrewAI is higher-level: you define roles and the framework handles orchestration- CrewAI is faster to prototype; LangGraph is better for production**Think of it as:** CrewAI = "hire a team" vs LangGraph = "design a workflow".</details>### Q3: What is DSPy and why is it different from other frameworks?<details><summary>💡 Answer</summary>**DSPy (Declarative Self-improving Python)** by Stanford NLP:Instead of writing prompts manually, you:1. Define the task as a **signature** (input → output types)2. Provide **examples** (training data)3. DSPy **compiles** the optimal prompt automatically```pythonclass RAG(dspy.Module):    def forward(self, question):        context = self.retrieve(question)        return self.generate(context, question)# Compile: finds optimal prompt through optimizationcompiled = dspy.BootstrapFewShot(metric=accuracy).compile(RAG())```**Key insight:** Treats prompt engineering as an **optimization problem**, not a writing problem. This is fundamentally different from all other frameworks.</details>### Q4: If you're designing a production RAG system for a large enterprise, which frameworks would you recommend and why?<details><summary>💡 Answer</summary>**Recommended stack:**1. **LlamaIndex** — Data ingestion layer   - 160+ connectors for diverse enterprise data (Confluence, Sharepoint, Slack, databases)   - Strong chunking and indexing strategies2. **LangChain + LangGraph** — Application & agent layer   - LCEL for chain composition   - LangGraph for stateful agent workflows with persistence   - Human-in-the-loop for sensitive operations3. **LangSmith** — Observability layer   - Tracing every LLM call   - Evaluation datasets for regression testing   - Cost monitoring4. **ChromaDB/Pinecone/Weaviate** — Vector storage   - Pinecone for production scale   - ChromaDB for developmentThis is the most common production stack in the industry.</details>

---## ✅ Notebook 22 Summary| Framework | Sweet Spot | Key Concept ||-----------|-----------|-------------|| **LangChain** | General LLM apps | LCEL chains, universal interface || **LangGraph** | Stateful agents | Graph-based state machines || **LangSmith** | Observability | Tracing, evaluation, monitoring || **LlamaIndex** | Data-heavy RAG | 160+ connectors, query engines || **CrewAI** | Multi-agent teams | Role-based agent modeling || **AutoGen** | Agent conversations | Conversational multi-agent || **DSPy** | Prompt optimization | Compile optimal prompts from data || **Haystack** | Enterprise NLP | Pipeline-based architecture |### 🏁 Phase 7 Complete — Move to [Capstone Projects](./capstone/) to build with these tools!